# Post-CPFE analysis & plotting

Load CPFE field outputs and plot field distributions, macroscopic
stress-strain, and per-block time series. This tutorial is pure Python (no
MOOSE/NEPER) and runs on the sample data under `mwe_data/`.

**Needs:** pip install only (a pole figure additionally needs NEML2).

The full script is `examples/demonstrate_postprocess.py`.

## Inputs

A CPFE block CSV plus a directory of per-time grid field CSVs. The repo ships
`mwe_data/out.csv` and `mwe_data/grid_out/`. If a run produced no `grid_out/`
CSVs (the CPFE defaults write them only with `grid_transfer="per_step"`),
regenerate them offline from the native Exodus with
`graintrace.grid_resampling.GridResampler` (needs `puma-opt`).

## Load results

In [ ]:
from graintrace.simulation_postprocessing import SimulationResults, FieldFileNaming
from graintrace import plot_postprocessing as postprocess

res = SimulationResults(
    block_csv="../../mwe_data/out.csv",
    field_dir="../../mwe_data/grid_out",
    field_naming=FieldFileNaming(
        prefix="out_element_centroid", index_width=4, sep="_", suffix=".csv"
    ),
)

## Plot distributions and macroscopic response

`tensor_prefix` selects the field (`ee`, `cauchy_stress`, `nye_tensor`,
`ori_rodrigues`, `strain`); `order` is the tensor rank.

In [ ]:
postprocess.plot_block_properties_distribution(
    res, time=1.0, tensor_prefix="ee", order=2, output_folder="out/post"
)
postprocess.plot_macroscopic_stress_strain(
    res,
    stress_tensor_prefix="cauchy_stress",
    strain_tensor_prefix="strain",
    volume_prefix="volume",
    output_folder="out/post",
)
postprocess.plot_block_properties_over_time(
    res, tensor_prefix="cauchy_stress", order=2, output_folder="out/post"
)

## Optional: pole figure and IPF coloring

A pole figure uses `neml2.texture` (NEML2 v3). IPF coloring writes RGB fields
onto an Exodus or VTK mesh via `graintrace.ipf_postprocess.IPFProcessor`.

In [ ]:
# Requires NEML2 v3 (neml2.texture):
# postprocess.plot_pole_figure(
#     res, tensor_prefix="ori_rodrigues", time=1.0, direction=[0, 0, 1],
#     crystal_symmetry="432", device="cpu", output_folder="out/post",
# )

## Gotchas

- Pre-load steps (t <= `initialize_time`) have near-zero stress/ee; that is
  expected.
- Pole figures require NEML2 v3; reinstall it if the `neml2.texture` import fails.
- The experiment half uses `experiment_postprocessing.ExperimentResults` with its
  own `FieldFileNaming` (e.g. prefix `expsyn_`, suffix `time.csv`).